In [56]:
import pandas as pd
from uuid import uuid4
from transformers import AutoTokenizer
from math import ceil
import requests
from bs4 import BeautifulSoup
import re
from collections import defaultdict


tokenizer = AutoTokenizer.from_pretrained(
    "McGill-NLP/LLM2Vec-Meta-Llama-31-8B-Instruct-mntp" ## adjust tokenization model to the one that is used in the embedding/retriever achitecture
)

token_length = 512 # adjust to maximal token length

# restricting legth (make room for cls token and paragraph seperators)
TOK_LEN = token_length - 10

In [57]:
# read  data
df = pd.read_parquet('/raid/deallab/SF_RAG_Data/ASQA/train.parquet')

for col in df.columns:
    print(col,':')
    print(df.loc[1, col], '\n')

ambiguous_question :
Who won the 2016 ncaa football national championship? 

qa_pairs :
[{'context': "The 13–1 Alabama Crimson Tide won the game, holding off the undefeated Clemson Tigers 45–40 in the fourth quarter. Accompanied by a talented receiving corps, Clemson's Heisman Finalist quarterback Deshaun Watson had a historic performance, setting the record for most total yards in national championship game history, with 478 yards (405 passing / 73 rushing) against the nation's third-ranked defense in Alabama, breaking the record previously set by Vince Young in the 2006 Rose Bowl. Following the game, the AP Poll also named Alabama as its top team of the season, giving Alabama their fourth title in seven seasons. Both Clemson and Alabama finished the season 14–1.", 'question': "Who won the 2016 season's ncaa football national championship?", 'short_answers': array(['Clemson Tigers', '2016 Clemson Tigers football team',
        '2016 Clemson Tigers football', 'the Tigers', 'Clemson',
 

In [58]:
#parse tables to text
def get_table(table):
    table_text = []
    for i, tr in enumerate(table.find('tbody').findChildren("tr" , recursive=False)):
        tr_text = tr.get_text()
        tr_text = re.sub(r'\n+',';',tr_text).strip(';')
        if not tr_text: continue
        if table_text == []:
            tr_text = '\n#### Table: ' + tr_text
        table_text.append(tr_text)

    return '\n'.join(table_text)

# parse pars to text
def get_p(par):
    p_text = par.get_text()
    p_text = p_text.replace('\n','')
    return p_text

def get_h(heading):
    h = heading.find(['h1', 'h2', 'h3','h4'])
    heading_type = int(re.search(r'<h(\d)', str(h)).group(1))
    h_text ='\n' + ' '.join(['#'*heading_type,h.get_text()])
    return h_text

#pars unordered lists to text
def get_ul(ul):
    list_text = []
    for li in ul.find_all('li'):
        list_text.append('* ' + li.get_text())
    return '\n'.join(list_text)

# pars ordered list to text
def get_ol(ol):
    list_text = []
    for i, li in enumerate(ol.find_all('li')):
        if li.get_text():
            list_text.append(' '.join([str(i+1),li.get_text().replace('\n','')]))
    return '\n'.join(list_text)
        

# parse whole document
def parse_document(doc):
    content = doc.find_all(['div', 'p', 'table', 'ul', 'ol'])
    document  = []
    for cont in content:
        #stop condition
        if cont.name == 'div' and cont.find(['h1', 'h2', 'h3','h4'], id=['See_also', 'References']):
            break
        
        #get headining
        if cont.name == 'div' and cont.has_attr('class') and  'mw-heading' in cont['class']:
            document.append(get_h(cont))
        # get par
        elif cont.name == 'p':
            par = get_p(cont)
            if par:
                document.append(par)
        #get ul
        elif cont.name == 'ul':
            document.append(get_ul(cont))
        #get ol
        elif cont.name == 'ol':
            document.append(get_ol(cont))
        #get table
        elif cont.name == 'table':
            if cont.has_attr('class') and 'metadata' in cont['class']: continue
            document.append(get_table(cont))
        # explore div
        elif cont.name == 'div':
            document.append(parse_document(cont))

    return '\n'.join(document).strip('\n')



In [59]:
for q in df.loc[1, 'qa_pairs']:
    print(q.keys())

dict_keys(['context', 'question', 'short_answers', 'wikipage'])
dict_keys(['context', 'question', 'short_answers', 'wikipage'])


In [72]:
# create embedding document dataset.
evidence_df = pd.DataFrame(columns=['id', 'sample_id','title', 'url', 'question', 'text'])

#creating question evidence pairs for retrival training (not implemented)
qe_df = pd.DataFrame(columns=['id', 'sample_id', 'question', 'evidence_id'])

#fetched_documents = {}
for idx, row in df.iterrows():
    if idx == 100: break
    sample_id = row['sample_id']
    evidences = row['wikipages']
    q1 = row['ambiguous_question']
    q2 = defaultdict(list)
    for q in row['qa_pairs']:
        question = q['question']
        wikipage = q['wikipage']
        if not question or not wikipage: continue
        q2[wikipage].append(question)
    for evidence in evidences:
        url = evidence['url']
        #if url in fetched_documents: continue
        #fetched_documents[url] = []
        page = requests.get(url)
        
        # Create a BeautifulSoup object
        soup = BeautifulSoup(page.text, 'html.parser')
        # get title
        title = soup.find(id='firstHeading').get_text()
        
        #extract content
        content = soup.find(class_='mw-content-ltr')
        parsed_doc = parse_document(content)
        
        # chunk document
        documents = [[]]
        
        for par in re.split(r'(?=\n#{1,4})', parsed_doc):
            tokenized_par = tokenizer.encode(par, add_special_tokens = False)
            length = len(tokenized_par)
            if len(documents[-1]) + length < TOK_LEN:
                documents[-1].extend(tokenized_par)
            elif length > TOK_LEN:
                begin = 0 
                while begin < length:
                    if begin + TOK_LEN >= length:
                        documents.append(tokenized_par[begin:])
                        break
                    documents.append(tokenized_par[begin:begin + TOK_LEN])
                    begin += TOK_LEN - int(TOK_LEN * 0.1)
            else:
                documents.append(tokenized_par)
            
        print(len(documents))
        for doc in documents:
            doc_text = tokenizer.decode(doc)
            id = uuid4()
            #fetched_documents[url].append(id)
            evidence_df.loc[len(evidence_df)] = [id, sample_id, title, url, q1, doc_text]
        
        if title in q2:
            for question in q2[title]:
                for doc in documents:
                    doc_text = tokenizer.decode(doc)
                    id = uuid4()
                    #fetched_documents[url].append(id)
                    evidence_df.loc[len(evidence_df)] = [id, sample_id, title, url, question, doc_text]
                
#print(fetched_documents)
evidence_df

158
16
14
11


,id,sample_id,title,url,question,text
0,886f3cc3-ab1a-4f99-903d-8e090b7114e9,-5742327688291876861,List of Bunk'd episodes,https://en.wikipedia.org/wiki/List%20of%20Bunk...,When does the new bunk'd come out?,Bunk'd is an American comedy television series...
1,6442ec3b-3b23-4010-97ce-567f92830e81,-5742327688291876861,List of Bunk'd episodes,https://en.wikipedia.org/wiki/List%20of%20Bunk...,When does the new bunk'd come out?,\n#### Table: SeasonEpisodesOriginally aired\n...
2,e35acc2d-cb64-4ed5-9220-4e04a95b2aa1,-5742327688291876861,List of Bunk'd episodes,https://en.wikipedia.org/wiki/List%20of%20Bunk...,When does the new bunk'd come out?,\n#### Table: No.overallNo. inseasonTitle [1][...
3,607d9d37-e5d8-44df-95da-ee2ecfc6a094,-5742327688291876861,List of Bunk'd episodes,https://en.wikipedia.org/wiki/List%20of%20Bunk...,When does the new bunk'd come out?,She assigns the counselors-in-training to giv...
4,76b359d1-6229-4050-9e3d-dc06a0d6b1a6,-5742327688291876861,List of Bunk'd episodes,https://en.wikipedia.org/wiki/List%20of%20Bunk...,When does the new bunk'd come out?,"as Gladys, Casey Campbell as Murphy;Absent: N..."
...,...,...,...,...,...,...
219,a29a2090-975c-41b8-9770-dce7ab8195e1,-3582047784487750233,2017 College Football Playoff National Champio...,https://en.wikipedia.org/wiki/2017%20College%2...,Who won the ncaa football national championshi...,-yard rush just a few plays later to give Alab...
220,807947f5-7761-4cd8-ac03-14f838ee2bf5,-3582047784487750233,2017 College Football Playoff National Champio...,https://en.wikipedia.org/wiki/2017%20College%2...,Who won the ncaa football national championshi...,\n#### Table: Quarter;1;2;34Total\nNo. 2 Clems...
221,c2322836-f367-42c4-a098-18b5bf938cbf,-3582047784487750233,2017 College Football Playoff National Champio...,https://en.wikipedia.org/wiki/2017%20College%2...,Who won the ncaa football national championshi...,\n#### Table: Scoring summary\nQuarter;Time;Dr...
222,81d1d174-3f04-4364-98c9-170e3a3a16e4,-3582047784487750233,2017 College Football Playoff National Champio...,https://en.wikipedia.org/wiki/2017%20College%2...,Who won the ncaa football national championshi...,\n#### Table: Quarter;Time;Drive;Team;Scoring ...


In [69]:
q2

defaultdict(list,
            {'2016 College Football Playoff National Championship': ["Who won the 2016 season's ncaa football national championship?"],
             '2017 College Football Playoff National Championship': ['Who won the ncaa football national championship played in 2016?']})

In [71]:
title in q2

True

In [12]:
# documents = [[]]

# for par in re.split(r'(?=\n#{1,4})', parsed_doc):
#     tokenized_par = tokenizer.encode(par, add_special_tokens = False)
#     length = len(tokenized_par)
#     if len(documents[-1]) + length < 1010:
#         documents[-1].extend(tokenized_par)
#     elif length > 1010:
#         begin = 0 
#         while begin < length:
#             if begin + 1010 >= length:
#                 documents.append(tokenized_par[begin:])
#                 break
#             documents.append(tokenized_par[begin:begin + 1010])
#             begin += 810
#     else:
#         documents.append([par])

# for doc in documents:
#     print(tokenizer.decode(doc))

#### Table: 2015  College Football Playoff National Championship presented by AT&T
Inaugural College Football Playoff National Championship
Ohio State Buckeyes;Oregon Ducks;(13–1);(13–1);Big Ten;Pac-12;42;20;Head coach: Urban Meyer;Head coach: Mark Helfrich;APCoachesCFP;544;APCoachesCFP;332
1234;Total;Ohio State;147714;42;Oregon;73100;20
DateJanuary 12, 2015
Season2014
StadiumAT&T Stadium
LocationArlington, Texas
MVPOffensive: #15 RB Ezekiel Elliott, So. Ohio StateDefensive: #23 S Tyvis Powell, So. Ohio State
FavoriteOregon by 7[1][2]
National anthemLady Antebellum[3]
RefereeGreg Burks (Big 12)
Attendance85,689
United States TV coverage
NetworkESPN[4][5]
AnnouncersChris Fowler, Kirk Herbstreit, Heather Cox and Tom Rinaldi (ESPN)Eduardo Varela and Pablo Viruega (ESPN Deportes)Mike Tirico, Todd Blackledge, Holly Rowe and Joe Schad (ESPN Radio)
Nielsen ratings18.9 (33.4 million viewers)
 College Football Playoff National Championship ;   ; 2016 >  ;College Football Championship Game;  < 2

TypeError: argument 'ids': 'str' object cannot be interpreted as an integer